# 06m — Generalization: Breakfast (action-segmentation family, Phase 2) — **PRIMARY**

PAPER_TODO §2.1. Breakfast (|V|~=48, N=1712, **10 activity classes**) is the closest external analog to
Smartflat (cooking actions, temporal phases) and the **only** action-seg dataset with enough videos per
class for the order-null — so it carries the **counterpoint to SDS2's 0/15**: *does symbol order
discriminate activities beyond frequency, on data with genuine recipe-phase ordering?*

**Bounds (reported honestly):** 10 classes -> C(10,2)=45 pairs x 4 grid combos is intractable, so the
order-null is run on the **top-6 activities by video count** (C(6,2)=15 comparisons) at `n_shuffles=200`
(apples-to-apples with SDS2). The quality half covers **all 10** activities on a **<=50/class subsample**
to bound the O(n^2) pairwise-rTWE that `k_medoid` triggers. Kernel: `smartflat_repro`.

In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault('NUMBA_THREADING_LAYER', 'workqueue')  # fork-safe rTWE under nbconvert
os.environ.setdefault('MPLBACKEND', 'agg')                   # headless figures
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display

from smartflat.utils.utils_io import get_data_root
from smartflat.utils.utils import upsample_sequence
from smartflat.features.symbolic_barycenter.generalization.action_segmentation import (
    download_action_seg, load_action_seg, build_action_seg_ground_cost,
    default_root, read_mapping, DATASETS)
from smartflat.features.symbolic_barycenter.generalization.suite import run_generalization_suite
from smartflat.features.symbolic_barycenter.registries import default_baseline_methods
from smartflat.features.symbolic_barycenter.visualization import plot_cohort_barycenters

NAME, UPSAMPLE = 'breakfast', 24   # UPSAMPLE ~= 2x median segment count (measured: median 6, max 25)
OUT = os.path.join(get_data_root(), 'outputs', 'symbolic_barycenter', 'generalization', NAME)
os.makedirs(OUT, exist_ok=True)
print('dataset:', NAME, '| upsample_to:', UPSAMPLE, '| output dir:', OUT)

dataset: breakfast | upsample_to: 24 | output dir: /home/perochon/data-gold-final/outputs/symbolic_barycenter/generalization/breakfast


## 1. Load + config-vs-real-files sanity

In [2]:
# GT action labels used DIRECTLY as symbols (RLE'd -> segment sequences, background -> 0).
# download_action_seg pulls only the tiny GT text via HTTP Range (never the 30 GB features);
# it is flag-guarded, so this is a no-op once the data is cached.
download_action_seg(NAME)
meta, X, labels, G = load_action_seg(NAME)
cfg = DATASETS[NAME]
ns = meta['n_segments'].to_numpy()
display(pd.DataFrame({
    'metric': ['N videos (loaded)', '|V| = G (incl. background 0)', '#activity classes',
               'segment length p10/50/90', 'published N', 'published #classes'],
    'value':  [len(X), G, len(set(labels)),
               tuple(np.percentile(ns, [10, 50, 90]).round(1)),
               cfg['n_videos'], cfg['n_classes']],
}))
print('activities:', sorted(set(labels)))
print('per-activity video counts:', dict(Counter(labels)))

,metric,value
0,N videos (loaded),1712
1,|V| = G (incl. background 0),48
2,#activity classes,10
3,segment length p10/50/90,"(4.0, 6.0, 11.0)"
4,published N,1712
5,published #classes,10


activities: ['cereals', 'coffee', 'friedegg', 'juice', 'milk', 'pancake', 'salat', 'sandwich', 'scrambledegg', 'tea']
per-activity video counts: {'cereals': 184, 'coffee': 167, 'friedegg': 173, 'milk': 187, 'salat': 163, 'sandwich': 169, 'tea': 184, 'pancake': 157, 'scrambledegg': 166, 'juice': 162}


## 2. Co-occurrence ground cost

In [3]:
# Data-driven co-occurrence ground cost over the symbol alphabet (symbols that frequently
# abut are closer) -> shared vocab.compute_distance_matrix. Built once; reused by the suite.
D_G = build_action_seg_ground_cost(NAME, X=X, kind='cooccurrence')
assert np.allclose(D_G, D_G.T) and np.allclose(np.diag(D_G), 0.0), 'D_G must be symmetric, zero-diagonal'
if D_G.shape != (G, G):   # loader G=len(mapping) vs ground-cost G=max(observed)+1 (a top id unused)
    print(f'NOTE: ground-cost G={D_G.shape[0]} != mapping G={G}; using {D_G.shape[0]} for shape-consistency')
    G = D_G.shape[0]
print('D_G shape:', D_G.shape, '| symmetric, zero-diagonal OK')

D_G shape: (48, 48) | symmetric, zero-diagonal OK


## 3. Order-null (headline) — top-6 activities, 15 pairs, 200 shuffles

Frequency-preserving order-shuffle null: `delta_auc = auc_intact - mean(auc_shuffled)` with a permutation
band; `order_helps` iff `ci_low > 0`. Reported as it lands — a positive OR null result is informative.

**Bound (probe-measured):** each of the top-6 activities is capped at **80 videos** (480 total). The full
3×5-fold CV × 200 shuffles × 15 pairs × 4 feature/shuffle combos over |V|=48 (2304-dim transition features)
is ~6.8 h of compute — kept in full for a rigorous, SDS2-comparable counterpoint (n_shuffles=200 unchanged).

In [4]:
# Cap each of the top-6 activities at 80 videos (probe-measured ~6.8 h at full 3x5-fold CV,
# n_shuffles=200 kept for apples-to-apples with SDS2's 0/15).
CAP_ORD, rng_o = 80, np.random.default_rng(1)
top6 = [a for a, _ in Counter(labels).most_common(6)]
idx6 = []
for a in top6:
    ia = np.where(labels == a)[0]
    idx6 += list(ia if len(ia) <= CAP_ORD else rng_o.choice(ia, CAP_ORD, replace=False))
idx6 = np.array(sorted(idx6))
X6, labels6 = [X[i] for i in idx6], labels[idx6]
print('top-6 activities:', top6, f'| capped N={len(X6)} (<= {CAP_ORD}/class):',
      {a: int((labels6 == a).sum()) for a in top6})
res_ord = run_generalization_suite(X6, labels6, G, D_G, name=NAME, run_quality=False, n_shuffles=200)
res_ord['order'].to_csv(os.path.join(OUT, 'order_null.csv'), index=False)
n_pos = int(res_ord['order']['order_helps'].sum())
print(f"\norder_helps (ci_low > 0): {n_pos}/{len(res_ord['order'])}   [SDS2 was 0/15]")
display(res_ord['order'][['comparison', 'feature', 'shuffle', 'auc_intact',
                          'delta_auc', 'ci_low', 'ci_high', 'p_perm', 'order_helps']].round(3))

top-6 activities: ['milk', 'cereals', 'tea', 'friedegg', 'sandwich', 'coffee'] | capped N=480 (<= 80/class): {'milk': 80, 'cereals': 80, 'tea': 80, 'friedegg': 80, 'sandwich': 80, 'coffee': 80}



order_helps (ci_low > 0): 0/60   [SDS2 was 0/15]


,comparison,feature,shuffle,auc_intact,delta_auc,ci_low,ci_high,p_perm,order_helps
0,cereals_vs_coffee,transition,token,1.0,0.0,0.0,0.0,1.00,False
1,cereals_vs_friedegg,transition,token,1.0,0.0,0.0,0.0,1.00,False
2,cereals_vs_milk,transition,token,1.0,0.0,0.0,0.0,1.00,False
3,cereals_vs_sandwich,transition,token,1.0,0.0,0.0,0.0,1.00,False
4,cereals_vs_tea,transition,token,1.0,0.0,0.0,0.0,1.00,False
5,coffee_vs_friedegg,transition,token,1.0,0.0,0.0,0.0,1.00,False
6,coffee_vs_milk,transition,token,1.0,0.0,0.0,0.0,0.99,False
7,coffee_vs_sandwich,transition,token,1.0,0.0,0.0,0.0,1.00,False
8,coffee_vs_tea,transition,token,1.0,0.0,0.0,0.0,1.00,False
9,friedegg_vs_milk,transition,token,1.0,0.0,0.0,0.0,1.00,False


## 4. Representation quality — all 10 activities, <=50/class subsample

In [5]:
CAP, rng = 50, np.random.default_rng(42)
idx = []
for a in sorted(set(labels)):
    ia = np.where(labels == a)[0]
    idx += sorted(ia if len(ia) <= CAP else rng.choice(ia, CAP, replace=False))
idx = np.array(sorted(idx))
Xq, labelsq = [X[i] for i in idx], labels[idx]
print(f'quality subsample: N={len(Xq)} (<= {CAP}/activity x {len(set(labels))} activities)')
res_q = run_generalization_suite(Xq, labelsq, G, D_G, name=NAME, run_order=False, upsample_to=UPSAMPLE)
res_q['quality'].reset_index().to_csv(os.path.join(OUT, 'quality.csv'), index=False)
display(res_q['quality'].round(3))

quality subsample: N=500 (<= 50/activity x 10 activities)


/home/perochon/anaconda3/envs/smartflat_repro/lib/python3.11/site-packages/ot/bregman/_barycenter.py:250: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


metric,inertia_rtwe,inertia_native,freq_fidelity,struct_preservation,entropy_bits,n_distinct,n_segments,stability_inertia_rtwe,stability_histogram,dataset
method,,,,,,,,,,
dba_dtw,4.521,6.641,0.177,1.752,2.184,5.65,7.8,0.295,0.004,breakfast
edit_median,3.432,7.458,0.126,1.369,2.182,5.00,6.5,0.000,0.000,breakfast
k_medoid,3.442,3.442,0.141,1.408,2.152,4.80,6.2,0.000,0.000,breakfast
majority_voting,3.663,0.366,0.154,1.435,2.124,5.00,6.7,0.000,0.000,breakfast
soft_dtw,4.239,-16.710,0.178,1.603,2.201,5.50,7.6,0.319,0.003,breakfast
wasserstein,NaN,0.164,0.095,NaN,2.321,48.00,NaN,NaN,0.000,breakfast


## 5. Prototypical execution per activity (chronograms)

In [6]:
# Prototypical execution per activity: a deterministic edit-median barycenter of each
# activity's executions, rendered as a chronogram strip (reuses plot_cohort_barycenters).
methods = default_baseline_methods(D_G)
code_to_label = {i: n for n, i in
                 read_mapping(os.path.join(default_root(NAME), 'mapping.txt'), cfg['background']).items()}
CAP_CHRONO, rng = 60, np.random.default_rng(0)
proto = {}
for a in sorted(set(labels)):
    ia = np.where(labels == a)[0]
    if len(ia) > CAP_CHRONO:
        ia = rng.choice(ia, CAP_CHRONO, replace=False)
    Xa = np.vstack([upsample_sequence(X[i], UPSAMPLE) for i in ia]).astype(int)
    proto[a] = np.asarray(methods['edit_median']['build'](Xa, 0)).astype(int)
plot_cohort_barycenters(proto, groups=sorted(proto), code_to_label=code_to_label, mask_background=True,
                        title=f'{NAME}: prototypical execution per activity (edit-median barycenter)',
                        savepath=os.path.join(OUT, 'chronograms.png'))
print('saved', os.path.join(OUT, 'chronograms.png'))

saved /home/perochon/data-gold-final/outputs/symbolic_barycenter/generalization/breakfast/chronograms.png


**Result.** `order_helps` = **0/60** (SDS2 was 0/15): symbol *order* adds no activity-discriminative
signal beyond frequency — the same conclusion as SDS2, now on genuinely procedural cooking data. In fact
`auc_intact = 1.0` on **all** 60 cells, so the top-6 activities are **perfectly separable by symbol
frequency alone** (which actions occur / how often); shuffling order — which preserves each sequence's
symbol multiset — leaves AUC at 1.0, hence `delta_auc = 0`.

**Honest caveat.** Frequency *saturates* here (AUC ceiling), so the ΔAUC null has no headroom to reveal
order signal even if some were present — a harder, frequency-controlled comparison (e.g. sub-activities
that share an action vocabulary) would be needed to probe order sensitively. The valid conclusion is that
order is *not required* for activity discrimination on Breakfast, consistent with SDS2.

**For the paper.** §7.2 evidence that the frequency-only result generalises externally; the quality table
feeds the §6.5 `tab:baselines` Breakfast column (edit-median / k-medoid the most faithful sequence
averagers; wasserstein the most frequency-faithful); chronograms show the prototypical recipe execution
per activity. CSVs: `order_null.csv`, `quality.csv`; figure `chronograms.png`.